In [1]:
import os
import json
from utils import set_seed, generate_namespace
from transformers import Trainer, TrainingArguments

from lora_fine_tune.model_local import LocalLM
from lora_fine_tune.data_utils import build_dataset

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
cfg = generate_namespace(path=f"../config.yaml")
print(json.dumps(vars(cfg), indent=2))

set_seed(cfg.seed)

{
  "model_name": "gpt2",
  "seed": 42,
  "max_token_length": 128,
  "lr": 0.0001,
  "warmup_ratio": 0.0,
  "epochs": 500,
  "train_batch_size": 1,
  "strategy": "steps",
  "logging_steps": 5,
  "res_path": "../results/"
}


In [3]:
model = LocalLM(cfg.model_name)

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 124,439,808
Trainable parameters: 124,439,808


In [4]:
model.lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [5]:
dataset = build_dataset()
data_tokenized = model.tokenize_dataset(dataset, cfg.max_token_length, padding=False)
data_collator = model.data_collator()
print(data_tokenized)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 5
})


In [6]:
training_args = TrainingArguments(
    output_dir=cfg.res_path,
    overwrite_output_dir=True,
    report_to="none",
    save_strategy="no",
    logging_strategy= "steps",
    logging_steps=cfg.logging_steps,
    learning_rate=cfg.lr,
    per_device_train_batch_size=cfg.train_batch_size,
    num_train_epochs=cfg.epochs,
    warmup_ratio=cfg.warmup_ratio,
    seed=cfg.seed
)

trainer = Trainer(
    model=model.lora_model,
    args=training_args,
    train_dataset=data_tokenized,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.900800
10,3.814900
15,3.795500
20,3.788700
25,3.833100
30,3.755400
35,3.428300
40,3.546600
45,3.334700
50,3.348200


TrainOutput(global_step=2500, training_loss=0.422285502576828, metrics={'train_runtime': 181.6452, 'train_samples_per_second': 13.763, 'train_steps_per_second': 13.763, 'total_flos': 27141590016000.0, 'train_loss': 0.422285502576828, 'epoch': 500.0})

In [7]:
eval_prompts = [
    "Q: How do transformers work?\nA:",
    "Q: What does physics explain?\nA:",
    "Q: What do machine learning models do?\nA:",
    "Q: What is a language model?\nA:",
    "Q: What is overfitting?\nA:",
    "Q: What is a LoRA adapter?\nA:",
]

for e in eval_prompts:
    print(f"Model output:\n{model.generate(e)}\n")
    print(f"LoRA model output:\n{model.generate(e, use_lora=True)}\n\n")

Model output:
Q: How do transformers work?
A: An inverted S is a function which takes three parameters, and multiplies it with the corresponding magnitude.

LoRA model output:
Q: How do transformers work?
A: Transformers process sequences using self-attention mechanisms. Transformers allow tokens in a sequence to attend each other


Model output:
Q: What does physics explain?
A: It doesn't necessarily have to be physical. You could say the same thing about quantum mechanics, and

LoRA model output:
Q: What does physics explain?
A: Physics explains natural phenomena using mathematical models. Model results show that objects in a sequence do not exist,


Model output:
Q: What do machine learning models do?
A: In the past, I've discussed what we're talking about here. We can model a whole range

LoRA model output:
Q: What do machine learning models do?
A: Machine-learning model predictions learn patterns from data. Model results can be self-attributing or


Model output:
Q: What is a lang